# 📊 Análisis con Pandas — Ejercicios con Soluciones

Este notebook cubre análisis de datos con pandas, visualización con matplotlib/seaborn y una introducción a Machine Learning.

**Cómo usar este notebook:**
1. Lee el **enunciado** y la **explicación conceptual**.
2. Intentá resolver en la celda `# TU CÓDIGO AQUÍ`.
3. Si necesitás ayuda, desplegá la **💡 Pista**.
4. Comparate con la **✅ Solución**.

**Librerías necesarias:**
```bash
pip install pandas matplotlib seaborn scikit-learn numpy
```
---

In [ ]:
# 🔧 CELDA DE SETUP — Ejecutá esto primero
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# Configuración visual general
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Librerías cargadas correctamente')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   matplotlib {matplotlib.__version__}')
print(f'   seaborn {sns.__version__}')

---
## 📦 Ejercicio 1 — Limpieza y análisis de ventas con pandas

**Enunciado:**
- Cargá un CSV de ventas con pandas.
- Eliminá filas con valores faltantes.
- Filtrá el dataset para mostrar solo las ventas de un producto específico.
- Calculá el total de ventas por mes y creá un gráfico de barras.

### 📖 Explicación conceptual

pandas trabaja con **DataFrames** (tablas). Las operaciones más comunes para limpiar datos son:

| Operación | Código | Descripción |
|-----------|--------|-------------|
| Ver valores nulos | `df.isnull().sum()` | Cuenta nulos por columna |
| Eliminar filas con nulos | `df.dropna()` | Borra filas con algún NaN |
| Rellenar nulos | `df.fillna(0)` | Reemplaza NaN por 0 |
| Filtrar filas | `df[df['col'] == valor]` | Selecciona filas que cumplen condición |
| Agrupar y sumar | `df.groupby('col')['val'].sum()` | Suma por grupo |

Para fechas, pandas tiene tipos especiales:
```python
df['fecha'] = pd.to_datetime(df['fecha'])  # Convierte texto a fecha
df['mes']   = df['fecha'].dt.month         # Extrae el mes
```

In [ ]:
# 🔧 GENERAMOS EL DATASET DE VENTAS CON VALORES FALTANTES
import numpy as np
import pandas as pd

np.random.seed(42)
n = 200

productos = ['Laptop', 'Auriculares', 'Mouse', 'Teclado', 'Monitor']
fechas    = pd.date_range('2023-01-01', periods=n, freq='2D')

df_ventas = pd.DataFrame({
    'fecha':    fechas,
    'producto': np.random.choice(productos, n),
    'cantidad': np.random.randint(1, 20, n),
    'precio':   np.random.uniform(10, 500, n).round(2),
    'vendedor': np.random.choice(['Ana', 'Carlos', 'María', None], n)  # Nulos en vendedor
})

# Introducimos algunos NaN artificiales
indices_nulos = np.random.choice(df_ventas.index, 15, replace=False)
df_ventas.loc[indices_nulos, 'cantidad'] = np.nan

df_ventas.to_csv('ventas_mensuales.csv', index=False)
print(f'Dataset creado: {df_ventas.shape[0]} filas, {df_ventas.shape[1]} columnas')
print(f'Valores nulos:\n{df_ventas.isnull().sum()}')
df_ventas.head()

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Cargá 'ventas_mensuales.csv' con pd.read_csv()
# 2. Mostrá cuántos valores nulos hay por columna
# 3. Eliminá las filas con valores nulos en 'cantidad'
# 4. Filtrá solo las ventas del producto 'Laptop'
# 5. Agregá una columna 'total' = cantidad * precio
# 6. Calculá el total de ventas por mes y graficalo


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

- Para limpiar: `df.dropna(subset=['cantidad'])` elimina solo filas donde 'cantidad' es nula.
- Para filtrar: `df[df['producto'] == 'Laptop']`
- Para extraer el mes: `pd.to_datetime(df['fecha'])` → luego `.dt.month_name()`
- Para graficar: `ventas_mes.plot(kind='bar')` o `sns.barplot()`

</details>

In [ ]:
# ✅ SOLUCIÓN

# 1. Cargamos el CSV
df = pd.read_csv('ventas_mensuales.csv', parse_dates=['fecha'])
print(f'Filas cargadas: {len(df)}')

# 2. Revisamos valores nulos
print('\nValores nulos por columna:')
print(df.isnull().sum())

# 3. Eliminamos filas con NaN en 'cantidad' (son las que no podemos calcular)
df_limpio = df.dropna(subset=['cantidad'])
print(f'\nFilas después de limpiar: {len(df_limpio)} (eliminadas: {len(df) - len(df_limpio)})')

# 4. Creamos columna total
df_limpio = df_limpio.copy()
df_limpio['total'] = df_limpio['cantidad'] * df_limpio['precio']

# 5. Filtramos solo Laptop
df_laptop = df_limpio[df_limpio['producto'] == 'Laptop']
print(f'\nRegistros de Laptop: {len(df_laptop)}')

# 6. Agrupamos por mes
df_limpio['mes'] = df_limpio['fecha'].dt.to_period('M')  # Período mensual
ventas_mes = df_limpio.groupby('mes')['total'].sum()

# 7. Gráfico de barras
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Total de ventas por mes (todos los productos)
ventas_mes.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Total de ventas por mes\n(todos los productos)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Mes')
axes[0].set_ylabel('Total de ventas ($)')
axes[0].tick_params(axis='x', rotation=45)

# Gráfico 2: Ventas por producto
ventas_producto = df_limpio.groupby('producto')['total'].sum().sort_values(ascending=False)
ventas_producto.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Total de ventas por producto', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Producto')
axes[1].set_ylabel('Total de ventas ($)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print('\n📊 Top 3 meses con más ventas:')
print(ventas_mes.sort_values(ascending=False).head(3))

---
## 📦 Ejercicio 2 — Análisis de películas

**Enunciado:**
- Cargá un dataset de películas (título, género, calificación).
- Encontrá el género más común.
- Calculá la calificación promedio por género.
- Creá un gráfico de barras de cantidad de películas por género.

### 📖 Explicación conceptual

Las operaciones de **agrupación** son el corazón del análisis con pandas:

```python
# groupby() + función de agregación
df.groupby('genero')['calificacion'].mean()   # Promedio por género
df.groupby('genero').size()                   # Conteo por género
df['genero'].value_counts()                   # Frecuencia de cada valor
df['genero'].value_counts().idxmax()          # El más frecuente
```

**value_counts()** es especialmente útil: devuelve cuántas veces aparece cada valor, ordenado de mayor a menor.

In [ ]:
# 🔧 GENERAMOS EL DATASET DE PELÍCULAS
np.random.seed(7)
n = 150

generos = ['Acción', 'Comedia', 'Drama', 'Terror', 'Ciencia Ficción', 'Animación', 'Romance']
# Hacemos que algunos géneros sean más frecuentes
pesos   = [0.25, 0.20, 0.18, 0.12, 0.10, 0.08, 0.07]

titulos = [f'Película {i+1}' for i in range(n)]

df_peliculas = pd.DataFrame({
    'titulo':        titulos,
    'genero':        np.random.choice(generos, n, p=pesos),
    'calificacion':  np.round(np.random.uniform(4.0, 10.0, n), 1),
    'año':           np.random.randint(2000, 2024, n),
    'duracion_min':  np.random.randint(75, 180, n)
})

df_peliculas.to_csv('peliculas.csv', index=False)
print(f'Dataset de películas creado: {df_peliculas.shape}')
df_peliculas.head()

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Cargá 'peliculas.csv'
# 2. Encontrá el género más común
# 3. Calculá la calificación promedio por género (ordenada de mayor a menor)
# 4. Graficá la cantidad de películas por género (barras)
# BONUS: Graficá también el promedio de calificación por género


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

- `df['genero'].value_counts()` → conteo por género (el primero es el más común)
- `df['genero'].value_counts().index[0]` → nombre del género más común
- `df.groupby('genero')['calificacion'].mean().sort_values(ascending=False)` → promedio ordenado
- Para graficar con seaborn: `sns.barplot(data=df, x='genero', y='calificacion', estimator='mean')`

</details>

In [ ]:
# ✅ SOLUCIÓN

df_p = pd.read_csv('peliculas.csv')

# 1. Género más común
conteo_generos = df_p['genero'].value_counts()
genero_mas_comun = conteo_generos.index[0]
print(f'🎬 Género más común: {genero_mas_comun} ({conteo_generos.iloc[0]} películas)')

# 2. Calificación promedio por género
prom_por_genero = df_p.groupby('genero')['calificacion'].mean().sort_values(ascending=False)
print('\n⭐ Calificación promedio por género:')
print(prom_por_genero.round(2))

# 3. Gráficos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Cantidad de películas por género
conteo_generos.plot(
    kind='bar', ax=axes[0],
    color=sns.color_palette('muted', len(conteo_generos)),
    edgecolor='white'
)
axes[0].set_title('Cantidad de películas por género', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Género')
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=30)
# Etiquetas sobre las barras
for bar in axes[0].patches:
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.3,
        int(bar.get_height()),
        ha='center', va='bottom', fontsize=10
    )

# Gráfico 2: Calificación promedio por género
prom_por_genero.plot(
    kind='bar', ax=axes[1],
    color=sns.color_palette('coolwarm', len(prom_por_genero)),
    edgecolor='white'
)
axes[1].set_title('Calificación promedio por género', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Género')
axes[1].set_ylabel('Calificación promedio')
axes[1].set_ylim(0, 10)
axes[1].axhline(df_p['calificacion'].mean(), color='red', linestyle='--', label=f'Media global ({df_p["calificacion"].mean():.1f})')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Tabla resumen completa
print('\n📋 Resumen por género:')
resumen = df_p.groupby('genero').agg(
    cantidad=('titulo', 'count'),
    calif_prom=('calificacion', 'mean'),
    calif_max=('calificacion', 'max'),
    duracion_prom=('duracion_min', 'mean')
).round(2).sort_values('cantidad', ascending=False)
print(resumen)

---
## 📦 Ejercicio 3 — Gráfico de líneas: precios de acciones

**Enunciado:** Usá datos históricos de precios de acciones y creá un gráfico de líneas que muestre la tendencia a lo largo del tiempo. Agregá etiquetas y título.

### 📖 Explicación conceptual

Los **gráficos de líneas** son ideales para datos temporales (series de tiempo). Con matplotlib:

```python
fig, ax = plt.subplots(figsize=(12, 5))   # Crea figura y ejes
ax.plot(x, y, label='Serie')              # Dibuja la línea
ax.set_title('Título')                    # Título del gráfico
ax.set_xlabel('Eje X')                    # Etiqueta eje X
ax.set_ylabel('Eje Y')                    # Etiqueta eje Y
ax.legend()                               # Muestra la leyenda
plt.xticks(rotation=45)                   # Rota etiquetas del eje X
plt.tight_layout()                        # Ajusta márgenes
plt.show()                                # Muestra el gráfico
```

También podés agregar **medias móviles** para suavizar la tendencia:
```python
df['media_movil'] = df['precio'].rolling(window=7).mean()  # Media de 7 días
```

In [ ]:
# 🔧 GENERAMOS DATOS SIMULADOS DE PRECIOS DE ACCIONES
np.random.seed(42)

fechas  = pd.date_range('2023-01-01', '2023-12-31', freq='B')  # 'B' = días hábiles
n_dias  = len(fechas)

# Simulamos un precio que camina aleatoriamente (random walk)
retornos_apple  = np.random.normal(0.0005, 0.015, n_dias)
retornos_google = np.random.normal(0.0003, 0.018, n_dias)

precio_apple  = 150 * np.cumprod(1 + retornos_apple)
precio_google = 100 * np.cumprod(1 + retornos_google)

df_acciones = pd.DataFrame({
    'fecha':  fechas,
    'APPLE':  precio_apple.round(2),
    'GOOGLE': precio_google.round(2),
    'volumen_apple':  np.random.randint(50_000_000, 150_000_000, n_dias)
})

df_acciones.to_csv('acciones.csv', index=False)
print(f'Dataset de acciones: {df_acciones.shape}')
print(df_acciones.head())

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Cargá 'acciones.csv' y convertí 'fecha' a datetime
# 2. Creá un gráfico de líneas para el precio de APPLE
# 3. Agregá la media móvil de 20 días
# 4. Poné título, etiquetas y leyenda
# BONUS: Graficá APPLE y GOOGLE en el mismo gráfico


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

- `pd.read_csv('acciones.csv', parse_dates=['fecha'])` → convierte la columna directamente.
- `df['APPLE'].rolling(window=20).mean()` → media móvil de 20 días.
- Para doble eje Y (precio y volumen): `ax2 = ax.twinx()`.
- Formatos de color útiles: `color='steelblue'`, `color='tomato'`, `linestyle='--'`.

</details>

In [ ]:
# ✅ SOLUCIÓN

df_acc = pd.read_csv('acciones.csv', parse_dates=['fecha'])

# Calculamos medias móviles
df_acc['MM_20_APPLE']  = df_acc['APPLE'].rolling(window=20).mean()
df_acc['MM_20_GOOGLE'] = df_acc['GOOGLE'].rolling(window=20).mean()

# --- Gráfico 1: APPLE con media móvil y volumen ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), gridspec_kw={'height_ratios': [3, 1]})

# Precio y media móvil
ax1.plot(df_acc['fecha'], df_acc['APPLE'],      color='steelblue', alpha=0.6,  linewidth=1,   label='Precio APPLE')
ax1.plot(df_acc['fecha'], df_acc['MM_20_APPLE'], color='navy',      linewidth=2, linestyle='--', label='Media móvil 20 días')

ax1.set_title('Evolución del precio de APPLE — 2023', fontsize=14, fontweight='bold')
ax1.set_ylabel('Precio (USD)', fontsize=11)
ax1.legend(fontsize=10)
ax1.tick_params(labelbottom=False)  # Ocultamos fechas del eje superior

# Sombreamos la zona por encima/debajo de la media
media_global = df_acc['APPLE'].mean()
ax1.axhline(media_global, color='gray', linestyle=':', alpha=0.7, label=f'Media global (${media_global:.0f})')

# Volumen (eje inferior)
ax2.bar(df_acc['fecha'], df_acc['volumen_apple'] / 1e6, color='steelblue', alpha=0.4, label='Volumen (M)')
ax2.set_ylabel('Volumen (millones)', fontsize=10)
ax2.set_xlabel('Fecha', fontsize=11)

plt.tight_layout()
plt.show()

# --- Gráfico 2: Comparación APPLE vs GOOGLE (normalizado al 100) ---
fig, ax = plt.subplots(figsize=(13, 5))

# Normalizamos al valor inicial = 100 para comparar rendimientos relativos
apple_norm  = (df_acc['APPLE']  / df_acc['APPLE'].iloc[0])  * 100
google_norm = (df_acc['GOOGLE'] / df_acc['GOOGLE'].iloc[0]) * 100

ax.plot(df_acc['fecha'], apple_norm,  color='steelblue', linewidth=2, label='APPLE')
ax.plot(df_acc['fecha'], google_norm, color='tomato',    linewidth=2, label='GOOGLE')
ax.axhline(100, color='gray', linestyle='--', alpha=0.5, label='Precio inicial (base 100)')

# Rellenamos la diferencia entre ambas
ax.fill_between(df_acc['fecha'], apple_norm, google_norm, alpha=0.1, color='purple')

ax.set_title('Rendimiento relativo APPLE vs GOOGLE — 2023\n(Base 100 = precio inicial)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Rendimiento relativo')
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Resumen numérico
for ticker in ['APPLE', 'GOOGLE']:
    rendimiento = ((df_acc[ticker].iloc[-1] / df_acc[ticker].iloc[0]) - 1) * 100
    print(f'{ticker}: rendimiento anual = {rendimiento:+.1f}%')

---
## 📦 Ejercicio 4 — Scatter plot con línea de tendencia

**Enunciado:** Usá datos de encuestas sobre preferencias alimenticias. Creá un scatter plot que relacione la edad con la preferencia por comida picante. Agregá una línea de tendencia.

### 📖 Explicación conceptual

Un **scatter plot** (diagrama de dispersión) muestra la relación entre dos variables numéricas. Cada punto es una observación.

La **línea de tendencia** (regresión lineal) muestra la dirección general de la relación:
- Pendiente positiva → a mayor X, mayor Y
- Pendiente negativa → a mayor X, menor Y
- Sin tendencia → las variables no están correlacionadas

```python
# Con seaborn es muy fácil agregar línea de tendencia:
sns.regplot(data=df, x='edad', y='picante', scatter_kws={'alpha': 0.5})
```

El **coeficiente de correlación** r mide la fuerza de la relación (-1 a 1):
```python
df['edad'].corr(df['picante'])  # Correlación de Pearson
```

In [ ]:
# 🔧 GENERAMOS EL DATASET DE ENCUESTA
np.random.seed(15)
n = 200

edad = np.random.randint(18, 70, n)
# Relación inversa: los más jóvenes prefieren más la comida picante
picante = np.clip(10 - (edad - 18) * 0.15 + np.random.normal(0, 2, n), 1, 10).round(1)

df_encuesta = pd.DataFrame({
    'edad':         edad,
    'picante':      picante,
    'genero':       np.random.choice(['M', 'F', 'Otro'], n, p=[0.48, 0.48, 0.04]),
    'region':       np.random.choice(['Norte', 'Centro', 'Sur'], n)
})

df_encuesta.to_csv('encuesta_picante.csv', index=False)
print(f'Encuesta generada: {df_encuesta.shape}')
print(df_encuesta.describe().round(2))

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Cargá 'encuesta_picante.csv'
# 2. Creá un scatter plot de edad vs picante
# 3. Agregá una línea de tendencia
# 4. Calculá e imprimí el coeficiente de correlación
# BONUS: Coloreá los puntos por género


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

- `sns.regplot(data=df, x='edad', y='picante')` hace el scatter + línea de tendencia en una sola línea.
- Para colorear por género: `sns.scatterplot(data=df, x='edad', y='picante', hue='genero')`.
- La correlación: `r = df['edad'].corr(df['picante'])`.
- Podés agregar texto al gráfico con: `ax.text(x, y, f'r = {r:.2f}')`.

</details>

In [ ]:
# ✅ SOLUCIÓN

df_enc = pd.read_csv('encuesta_picante.csv')

# Correlación de Pearson
r = df_enc['edad'].corr(df_enc['picante'])
print(f'Correlación edad-picante: r = {r:.3f}')
print(f'Interpretación: {"relación inversa moderada" if r < -0.3 else "poca o ninguna relación"}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: regplot (scatter + línea de tendencia automática)
sns.regplot(
    data=df_enc, x='edad', y='picante', ax=axes[0],
    scatter_kws={'alpha': 0.5, 'color': 'steelblue', 's': 40},
    line_kws={'color': 'tomato', 'linewidth': 2}
)
axes[0].set_title('Edad vs Preferencia por comida picante\n(con línea de tendencia)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Edad (años)')
axes[0].set_ylabel('Nivel de preferencia por picante (1-10)')
# Anotamos el coeficiente de correlación en el gráfico
axes[0].text(0.05, 0.95, f'r = {r:.3f}', transform=axes[0].transAxes,
             fontsize=12, color='tomato', fontweight='bold',
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Gráfico 2: scatter coloreado por género
sns.scatterplot(
    data=df_enc, x='edad', y='picante',
    hue='genero', style='genero', ax=axes[1],
    alpha=0.7, s=60, palette='Set2'
)
# Línea de tendencia manual para cada género
for genero, color in zip(['M', 'F'], ['steelblue', 'tomato']):
    sub = df_enc[df_enc['genero'] == genero]
    z   = np.polyfit(sub['edad'], sub['picante'], 1)
    p   = np.poly1d(z)
    x_line = np.linspace(sub['edad'].min(), sub['edad'].max(), 100)
    axes[1].plot(x_line, p(x_line), color=color, linewidth=1.5, linestyle='--')

axes[1].set_title('Edad vs Picante por género', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Edad (años)')
axes[1].set_ylabel('Preferencia por picante')

plt.tight_layout()
plt.show()

---
## 📦 Ejercicio 5 — Regresión lineal: precio de viviendas

**Enunciado:** Implementá un modelo de regresión lineal para predecir el precio de una vivienda en función de su tamaño. Evalualo con MSE.

### 📖 Explicación conceptual

La **regresión lineal** busca la recta que mejor describe la relación entre dos variables:
```
precio = m * tamaño + b
```

**Flujo de trabajo con scikit-learn** (siempre el mismo):
```python
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 1. Separar features (X) y target (y)
X = df[['tamaño']]  # Siempre 2D para sklearn
y = df['precio']

# 2. Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Crear y entrenar el modelo
modelo = LinearRegression()
modelo.fit(X_train, y_train)   # Aprende de los datos de entrenamiento

# 4. Predecir y evaluar
y_pred = modelo.predict(X_test)
mse    = mean_squared_error(y_test, y_pred)
```

| Métrica | Fórmula | Qué mide |
|---------|---------|----------|
| MSE | promedio de (real - predicho)² | Error promedio al cuadrado |
| RMSE | √MSE | Error en las mismas unidades que y |
| R² | 0 a 1 | Qué porcentaje de la variación explica el modelo |

In [ ]:
# 🔧 GENERAMOS EL DATASET DE VIVIENDAS
np.random.seed(42)
n = 300

tamaño   = np.random.uniform(40, 250, n)                          # m²
precio   = 1200 * tamaño + np.random.normal(0, 20000, n) + 50000  # relación lineal + ruido

df_casas = pd.DataFrame({
    'tamaño_m2': tamaño.round(1),
    'precio_usd': precio.round(0)
})

df_casas.to_csv('viviendas.csv', index=False)
print(f'Dataset de viviendas: {df_casas.shape}')
df_casas.describe().round(0)

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Cargá 'viviendas.csv'
# 2. Definí X (tamaño_m2) e y (precio_usd)
# 3. Dividí en train (80%) y test (20%) con train_test_split
# 4. Entrenás LinearRegression y predecís sobre el test
# 5. Calculá MSE, RMSE y R²
# 6. Graficá los puntos reales vs la recta de regresión


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

- `X = df[['tamaño_m2']]` → doble corchete para que sea 2D (sklearn lo requiere).
- Después de `modelo.fit()`, podés ver la pendiente con `modelo.coef_[0]` y el intercepto con `modelo.intercept_`.
- `RMSE = np.sqrt(MSE)` → te da el error en dólares.
- Para la recta: generá puntos con `np.linspace(X.min(), X.max(), 100)` y predecís sobre ellos.

</details>

In [ ]:
# ✅ SOLUCIÓN
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

df_c = pd.read_csv('viviendas.csv')

# 1. Features y target
X = df_c[['tamaño_m2']]   # 2D: doble corchete
y = df_c['precio_usd']

# 2. División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train)} muestras | Test: {len(X_test)} muestras')

# 3. Crear y entrenar el modelo
modelo = LinearRegression()
modelo.fit(X_train, y_train)   # .fit() = "aprende"

# 4. Predecir
y_pred = modelo.predict(X_test)

# 5. Métricas de evaluación
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print(f'\n📐 Ecuación del modelo:')
print(f'   Precio = {modelo.coef_[0]:.0f} × tamaño + {modelo.intercept_:.0f}')
print(f'\n📊 Métricas de evaluación:')
print(f'   MSE:  {mse:,.0f}')
print(f'   RMSE: ${rmse:,.0f} (error promedio en dólares)')
print(f'   R²:   {r2:.4f} → el modelo explica el {r2*100:.1f}% de la variación')

# 6. Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter + recta de regresión
x_linea = np.linspace(X.min().iloc[0], X.max().iloc[0], 100).reshape(-1, 1)
y_linea = modelo.predict(x_linea)

axes[0].scatter(X_train, y_train, alpha=0.4, color='steelblue', s=25, label='Train')
axes[0].scatter(X_test,  y_test,  alpha=0.6, color='orange',    s=30, label='Test')
axes[0].plot(x_linea, y_linea, color='tomato', linewidth=2.5, label=f'Regresión (R²={r2:.2f})')
axes[0].set_title('Regresión lineal: Tamaño vs Precio', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Tamaño (m²)')
axes[0].set_ylabel('Precio (USD)')
axes[0].legend()

# Valores reales vs predichos
axes[1].scatter(y_test, y_pred, alpha=0.5, color='steelblue', s=30)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[1].plot(lims, lims, 'r--', linewidth=2, label='Predicción perfecta')
axes[1].set_title('Valores reales vs predichos', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Precio real (USD)')
axes[1].set_ylabel('Precio predicho (USD)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 📦 Ejercicio 6 — Clasificación k-NN con dataset Iris

**Enunciado:** Implementá un modelo k-Nearest Neighbors para clasificar especies de flores. Evalualo con precisión y matriz de confusión.

### 📖 Explicación conceptual

**k-NN (k-Nearest Neighbors)** es un algoritmo de clasificación intuitivo:
> "Para clasificar un punto nuevo, mirá sus k vecinos más cercanos y votá por mayoría."

El **dataset Iris** es el "Hola Mundo" del ML: 150 flores con 4 medidas y 3 especies.

| Término | Significado |
|---------|-------------|
| Feature | Variable de entrada (ej: largo del sépalo) |
| Target/Label | Lo que queremos predecir (especie) |
| Train set | Datos para entrenar el modelo |
| Test set | Datos para evaluar (el modelo no los vio) |
| Accuracy | % de predicciones correctas |
| Matriz de confusión | Tabla de verdaderos/falsos positivos por clase |

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Cargá el dataset Iris con: from sklearn.datasets import load_iris
# 2. Dividí en train/test
# 3. Entrenás KNeighborsClassifier con k=5
# 4. Calculá la accuracy
# 5. Graficá la matriz de confusión
# BONUS: Probá distintos valores de k y graficá la accuracy en función de k


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```python
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

iris = load_iris()
X, y = iris.data, iris.target
```
Para la matriz de confusión: `sns.heatmap(confusion_matrix(y_test, y_pred), annot=True)`

</details>

In [ ]:
# ✅ SOLUCIÓN
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

# 1. Cargamos el dataset
iris    = load_iris()
X, y   = iris.data, iris.target
nombres = iris.target_names   # ['setosa', 'versicolor', 'virginica']

print(f'Dataset Iris: {X.shape[0]} muestras, {X.shape[1]} features')
print(f'Features: {iris.feature_names}')
print(f'Clases:   {list(nombres)}')

# 2. Escalamos los datos (importante para k-NN: usa distancias)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

# 4. Entrenamos con k=5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'\n✅ Accuracy con k=5: {accuracy:.4f} ({accuracy*100:.1f}%)')

print('\n📋 Reporte completo por clase:')
print(classification_report(y_test, y_pred, target_names=nombres))

# 5. Gráficos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=nombres, yticklabels=nombres,
    linewidths=0.5
)
axes[0].set_title('Matriz de confusión\nk-NN (k=5)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicho')

# Accuracy vs k
k_valores   = range(1, 31)
accuracies  = []
for k in k_valores:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train, y_train)
    acc = accuracy_score(y_test, knn_k.predict(X_test))
    accuracies.append(acc)

k_optimo = k_valores[np.argmax(accuracies)]
axes[1].plot(k_valores, accuracies, marker='o', color='steelblue', linewidth=2)
axes[1].axvline(k_optimo, color='tomato', linestyle='--', label=f'k óptimo = {k_optimo}')
axes[1].set_title('Accuracy vs valor de k', fontsize=12, fontweight='bold')
axes[1].set_xlabel('k (número de vecinos)')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].set_ylim(0.8, 1.01)

plt.tight_layout()
plt.show()

print(f'\n🏆 k óptimo: {k_optimo} (accuracy: {max(accuracies):.4f})')

---
## 📦 Ejercicio 7 — Proyecto: análisis de datos climáticos

**Enunciado:** Realizá un análisis completo de datos climáticos que incluya limpieza, visualizaciones y detección de patrones.

### 📖 Explicación conceptual

Un **análisis exploratorio de datos (EDA)** típico sigue estos pasos:

1. **Cargar y entender** → shape, dtypes, head(), describe()
2. **Limpiar** → nulos, duplicados, outliers
3. **Explorar distribuciones** → histogramas, boxplots
4. **Explorar relaciones** → scatter, correlación
5. **Explorar tendencias** → agrupaciones por tiempo/categoría
6. **Conclusiones** → hallazgos principales

In [ ]:
# 🔧 GENERAMOS EL DATASET CLIMÁTICO
np.random.seed(99)

fechas     = pd.date_range('2020-01-01', '2023-12-31', freq='D')
n          = len(fechas)
dia_año    = np.arange(n) % 365

# Temperatura con estacionalidad
temp_base  = 15 + 12 * np.sin(2 * np.pi * dia_año / 365 - np.pi/2)
temperatura = temp_base + np.random.normal(0, 3, n)

df_clima = pd.DataFrame({
    'fecha':         fechas,
    'temperatura_c': temperatura.round(1),
    'humedad_pct':   np.clip(60 - 0.3 * temperatura + np.random.normal(0, 10, n), 20, 100).round(1),
    'lluvia_mm':     np.where(np.random.rand(n) > 0.7, np.random.exponential(5, n), 0).round(1),
    'ciudad':        np.random.choice(['Buenos Aires', 'Córdoba', 'Mendoza'], n)
})

# Introducimos algunos nulos
df_clima.loc[np.random.choice(df_clima.index, 30), 'temperatura_c'] = np.nan

df_clima.to_csv('datos_climaticos.csv', index=False)
print(f'Dataset climático: {df_clima.shape}')
df_clima.head()

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Realizá un EDA completo:
# 1. Cargá y revisá el dataset (shape, info, describe, nulos)
# 2. Limpiá los valores nulos (podés rellenarlos con la media o interpolación)
# 3. Graficá la temperatura a lo largo del tiempo
# 4. Calculá el promedio mensual de temperatura y graficalo
# 5. Creá un boxplot de temperatura por ciudad
# 6. Analizá la correlación entre temperatura y humedad


In [ ]:
# ✅ SOLUCIÓN — EDA Completo

df_clim = pd.read_csv('datos_climaticos.csv', parse_dates=['fecha'])

# --- PASO 1: ENTENDER EL DATASET ---
print('=== PASO 1: Estructura del dataset ===')
print(f'Shape: {df_clim.shape}')
print(f'Período: {df_clim["fecha"].min().date()} → {df_clim["fecha"].max().date()}')
print(f'Nulos:\n{df_clim.isnull().sum()}')

# --- PASO 2: LIMPIEZA ---
# Interpolación lineal para temperatura (mejor que la media para series de tiempo)
df_clim['temperatura_c'] = df_clim['temperatura_c'].interpolate(method='linear')
print(f'\nNulos después de limpiar: {df_clim.isnull().sum().sum()}')

# Columnas derivadas útiles
df_clim['año']       = df_clim['fecha'].dt.year
df_clim['mes']       = df_clim['fecha'].dt.month
df_clim['mes_nombre'] = df_clim['fecha'].dt.strftime('%b')  # 'Jan', 'Feb', etc.

print('\n=== PASO 3-6: Visualizaciones ===')

fig = plt.figure(figsize=(16, 12))

# Gráfico 1: Serie de tiempo de temperatura
ax1 = fig.add_subplot(2, 2, 1)
# Media móvil de 30 días para ver la tendencia
temp_mm30 = df_clim.groupby('fecha')['temperatura_c'].mean().rolling(30).mean()
ax1.plot(df_clim['fecha'], df_clim['temperatura_c'], alpha=0.2, color='steelblue', linewidth=0.5)
ax1.plot(temp_mm30.index, temp_mm30.values, color='tomato', linewidth=2, label='Media móvil 30 días')
ax1.set_title('Temperatura a lo largo del tiempo', fontweight='bold')
ax1.set_ylabel('Temperatura (°C)')
ax1.legend()

# Gráfico 2: Temperatura promedio por mes
ax2 = fig.add_subplot(2, 2, 2)
meses_orden = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
prom_mensual = df_clim.groupby('mes_nombre')['temperatura_c'].mean()
prom_mensual = prom_mensual.reindex(meses_orden)
bars = ax2.bar(prom_mensual.index, prom_mensual.values,
               color=plt.cm.RdYlBu_r([v/prom_mensual.max() for v in prom_mensual.values]))
ax2.set_title('Temperatura promedio por mes', fontweight='bold')
ax2.set_ylabel('Temperatura (°C)')
ax2.tick_params(axis='x', rotation=45)

# Gráfico 3: Boxplot por ciudad
ax3 = fig.add_subplot(2, 2, 3)
sns.boxplot(data=df_clim, x='ciudad', y='temperatura_c', ax=ax3, palette='Set2')
ax3.set_title('Distribución de temperatura por ciudad', fontweight='bold')
ax3.set_xlabel('Ciudad')
ax3.set_ylabel('Temperatura (°C)')

# Gráfico 4: Correlación temperatura vs humedad
ax4 = fig.add_subplot(2, 2, 4)
sns.scatterplot(data=df_clim.sample(500, random_state=1), x='temperatura_c', y='humedad_pct',
                hue='ciudad', alpha=0.6, s=20, ax=ax4)
r = df_clim['temperatura_c'].corr(df_clim['humedad_pct'])
ax4.set_title(f'Temperatura vs Humedad (r={r:.2f})', fontweight='bold')
ax4.set_xlabel('Temperatura (°C)')
ax4.set_ylabel('Humedad (%)')

plt.suptitle('📊 EDA — Datos Climáticos 2020-2023', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n=== CONCLUSIONES ===')
print(f'Temperatura promedio global: {df_clim["temperatura_c"].mean():.1f}°C')
print(f'Mes más frío:    {prom_mensual.idxmin()} ({prom_mensual.min():.1f}°C)')
print(f'Mes más cálido:  {prom_mensual.idxmax()} ({prom_mensual.max():.1f}°C)')
print(f'Correlación temperatura-humedad: {r:.3f} (\'inversa moderada\')')

---
## 📦 Ejercicio 8 — Análisis de sentimiento (NLP básico)

**Enunciado:** Usá técnicas de NLP para determinar la polaridad (positivo/negativo/neutro) de reseñas de productos. Presentá los resultados visualmente.

### 📖 Explicación conceptual

El **análisis de sentimiento** clasifica texto según su tono emocional.

**Enfoque basado en léxico** (sin ML): se usa un diccionario de palabras positivas y negativas. Es simple pero efectivo para empezar.

**Enfoque con TextBlob**: biblioteca que asigna un `polarity` score de -1 (negativo) a +1 (positivo):
```python
from textblob import TextBlob
TextBlob('This is great!').sentiment.polarity   # → 1.0
TextBlob('This is terrible').sentiment.polarity  # → -1.0
```

**Preprocesamiento de texto** antes de analizar:
- Convertir a minúsculas
- Eliminar puntuación y caracteres especiales
- Tokenizar (dividir en palabras)

In [ ]:
# 🔧 DATASET DE RESEÑAS
resenas = [
    ("Excelente producto, muy buena calidad y llegó rápido",       5),
    ("Pésimo, se rompió al mes de uso, no lo recomiendo",          1),
    ("Está bien, cumple su función aunque esperaba más",            3),
    ("Increíble relación precio-calidad, volvería a comprarlo",     5),
    ("Llegó tarde y el empaque estaba dañado, decepcionante",       2),
    ("Ni bueno ni malo, un producto del montón",                    3),
    ("Fantástico, superó todas mis expectativas, altamente recomendable", 5),
    ("Terrible experiencia, el servicio al cliente fue horrible",   1),
    ("Buen producto aunque el manual de instrucciones es confuso",  4),
    ("Compra obligatoria si buscás calidad y durabilidad",          5),
    ("Regular, hay opciones mejores en el mercado",                 3),
    ("Me arrepiento de haberlo comprado, una pérdida de dinero",    1),
    ("Buena compra, el producto es como se describe",               4),
    ("Aceptable para el precio que tiene, nada del otro mundo",     3),
    ("Excelente atención y producto de primera calidad",            5),
]

df_resenas = pd.DataFrame(resenas, columns=['resena', 'estrellas'])
print(f'Dataset de reseñas: {df_resenas.shape}')
df_resenas

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Usaremos un enfoque de léxico (sin dependencias externas de ML)
# 1. Definí listas de palabras positivas y negativas en español
# 2. Creá una función que cuente palabras positivas y negativas en cada reseña
# 3. Asigná un sentimiento: 'Positivo', 'Negativo' o 'Neutro'
# 4. Graficá la distribución de sentimientos con un gráfico de torta
# 5. Comparás el sentimiento detectado con las estrellas reales


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

- Convertí cada reseña a minúsculas con `.lower()` y dividila en palabras con `.split()`.
- Contá cuántas palabras de cada reseña están en tu lista de palabras positivas/negativas.
- Si positivas > negativas → 'Positivo', si negativas > positivas → 'Negativo', si empate → 'Neutro'.
- Para el gráfico de torta: `df['sentimiento'].value_counts().plot(kind='pie')`.

</details>

In [ ]:
# ✅ SOLUCIÓN — Análisis de Sentimiento con Léxico en Español
import re

# Léxico básico en español
palabras_positivas = {
    'excelente','buena','bueno','fantástico','increíble','recomendable','calidad',
    'rápido','superó','obligatoria','durabilidad','primera','compra','bien','aceptable',
    'atención','altamente','relación'
}
palabras_negativas = {
    'pésimo','rompió','decepcionante','terrible','horrible','arrepiento','pérdida',
    'tarde','dañado','confuso','regular','malo','peor','arrepentimiento','montón'
}

def limpiar_texto(texto):
    """Convierte a minúsculas y elimina puntuación."""
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)  # Elimina puntuación
    return texto.split()                   # Devuelve lista de palabras

def analizar_sentimiento(resena):
    """Analiza el sentimiento de una reseña usando léxico."""
    palabras = limpiar_texto(resena)
    pos = sum(1 for p in palabras if p in palabras_positivas)
    neg = sum(1 for p in palabras if p in palabras_negativas)
    score = pos - neg
    if score > 0:
        return 'Positivo', score
    elif score < 0:
        return 'Negativo', score
    else:
        return 'Neutro', 0

# Aplicamos el análisis a todas las reseñas
resultados = df_resenas['resena'].apply(analizar_sentimiento)
df_resenas['sentimiento'] = [r[0] for r in resultados]
df_resenas['score']       = [r[1] for r in resultados]

print('📋 Reseñas con sentimiento detectado:')
print(df_resenas[['resena', 'estrellas', 'sentimiento', 'score']].to_string())

# Visualizaciones
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Gráfico 1: Distribución de sentimientos (torta)
conteo = df_resenas['sentimiento'].value_counts()
colores = {'Positivo': '#4CAF50', 'Neutro': '#FFC107', 'Negativo': '#F44336'}
axes[0].pie(
    conteo, labels=conteo.index, autopct='%1.0f%%',
    colors=[colores[s] for s in conteo.index],
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Distribución de sentimientos', fontweight='bold')

# Gráfico 2: Estrellas por sentimiento (boxplot)
orden = ['Positivo', 'Neutro', 'Negativo']
colores_lista = [colores[s] for s in orden if s in df_resenas['sentimiento'].unique()]
sns.boxplot(
    data=df_resenas, x='sentimiento', y='estrellas',
    order=[o for o in orden if o in df_resenas['sentimiento'].unique()],
    palette=colores, ax=axes[1]
)
axes[1].set_title('Estrellas por sentimiento detectado', fontweight='bold')
axes[1].set_ylabel('Estrellas (1-5)')
axes[1].set_xlabel('Sentimiento')

# Gráfico 3: Score de sentimiento vs estrellas
colores_puntos = [colores[s] for s in df_resenas['sentimiento']]
axes[2].scatter(df_resenas['estrellas'], df_resenas['score'],
                c=colores_puntos, s=100, edgecolor='white', linewidth=0.5)
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_title('Score léxico vs estrellas', fontweight='bold')
axes[2].set_xlabel('Estrellas reales')
axes[2].set_ylabel('Score de sentimiento')
axes[2].set_xticks([1, 2, 3, 4, 5])

from matplotlib.patches import Patch
leyenda = [Patch(color=c, label=s) for s, c in colores.items()]
axes[2].legend(handles=leyenda, loc='upper left')

plt.suptitle('📊 Análisis de Sentimiento — Reseñas de Productos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Métricas de accuracy del análisis
df_resenas['sentimiento_real'] = df_resenas['estrellas'].apply(
    lambda x: 'Positivo' if x >= 4 else ('Negativo' if x <= 2 else 'Neutro')
)
acc = (df_resenas['sentimiento'] == df_resenas['sentimiento_real']).mean()
print(f'\n✅ Precisión del análisis léxico: {acc:.0%}')

---
## 🏆 Resumen del notebook

| Ejercicio | Tema | Herramientas |
|-----------|------|--------------|
| 1 | Limpieza y análisis de ventas | `dropna`, `groupby`, `plot(kind='bar')` |
| 2 | Análisis de películas | `value_counts`, `groupby + agg`, `sns.barplot` |
| 3 | Serie de tiempo de acciones | `rolling`, `plot`, `fill_between` |
| 4 | Scatter + línea de tendencia | `sns.regplot`, `corr()`, `np.polyfit` |
| 5 | Regresión lineal | `LinearRegression`, MSE, RMSE, R² |
| 6 | Clasificación k-NN | `KNeighborsClassifier`, confusion matrix, accuracy |
| 7 | EDA climático completo | Interpolación, subplots, boxplot, heatmap |
| 8 | Análisis de sentimiento | Léxico, `re`, visualización con pie chart |

### 💪 Desafíos extra
1. **Ej. 1:** Agregá una columna `trimestre` y analizá las ventas por trimestre.
2. **Ej. 5:** Agregá más features (ej: habitaciones, edad de la vivienda) y usá regresión múltiple.
3. **Ej. 6:** Probá otros clasificadores: `RandomForestClassifier`, `SVC` y comparalos.
4. **Ej. 8:** Ampliá el léxico con más palabras y medí cómo mejora la precisión.